# Aerosol measurements at Mt. Kenya
Read an plot the prepared (level 2) aerosol measurements 

In [ ]:
# import 
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import pyplot
import os
import matplotlib.ticker as ticker
import numpy as np
import xarray as xr
import sys
from utils.utilities import find_best_grid_point, get_station_coords,form_xdate, get_anomalies
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import datetime as dt
from os import PathLike
from pathlib import Path
import zipfile

from plotting import tol_colors # color schemes from https://personal.sron.nl/~pault/
from utils import process_data
from input import read_aerosols

#activate interactive figures
%matplotlib widget
#activate autoreload
%load_ext autoreload

## Add parent directory to syspath
parent_dir = os.path.abspath(os.path.join(os.path.dirname('.'), '..'))
if not parent_dir in sys.path:
    sys.path.append(parent_dir)

#save figures in...
dir_save = './output/aerosols/'

### Read the data

In [ ]:
# Read in aerosol data from aethalometer and nephelometer
aerosol_data_dir = "..\data\level2\L2_AEROSOL_data_bachelorthesis Mike Baumann"
ds_ae, ds_neph = read_aerosols.aerosol_data_to_dataset(aerosol_data_dir)

ds_aerosols = read_aerosols.aerosol_to_dataset(ds_ae, ds_neph)
ds_aerosols

### Plotting

In [ ]:
## Absorption and scattering timeseries

fig, axs = plt.subplots(3,1,sharex=True)
# Total absorption coefficient at 521nm
ds_sel = ds_aerosols.sel(lambda_abs=521)['abs_coeff']
ax = axs[0]
ds_sel.plot(
    label=ds_sel.attrs["description"],
    ax=ax
)
ax.set_ylabel(f"{ds_sel.name} ({ds_sel.attrs["units"]})")
ax.set_title('')
ax.set_title('Total absorption coefficient at 521nm ',loc='left')
ax.set_xlabel('')

# Total Scattering coefficient at 525nm (green)
ds_sel = ds_aerosols.sel(lambda_scat=525)['scat_coeff']
ax = axs[1]
ds_sel.plot(
    label=ds_sel.attrs["description"],
    ax=ax
)
ax.set_ylabel(f"{ds_sel.name} ({ds_sel.attrs["units"]})")
ax.set_title('')
ax.set_title('Total scattering coefficient at 525nm',loc='left')

# # Equivalent black carbon
ds_sel = ds_aerosols.sel(lambda_abs=880)['black_carbon']
ax = axs[2]
ds_sel.plot(
    label=ds_sel.attrs["description"],
    ax=ax
)
ax.set_title('')
ax.set_ylabel(f"{ds_sel.name} ({ds_sel.attrs["units"]})") # I think the unit should be microgramm/m3
ax.set_title('Total equivalent black carbon',loc='left')

plt.show()

In [ ]:
## Seasonal cycles as plotted by Mike

freq = 'month' #"dayofyear"  #'month'
fig, axs = plt.subplots(2,1,sharex=True)


# Total absorption coefficient at 521nm
ds_sel = ds_aerosols.sel(lambda_abs=521)['abs_coeff']
ax = axs[0]
ds_sel.groupby(f"time.{freq}").mean().plot(
    label=ds_sel.attrs["description"],
    ax=axs[0]
)
ax.set_title('Total absorption coefficient at 521nm ',loc='left')
ax.set_xlabel('')

# Total Scattering coefficient at 525nm (green)
ds_sel = ds_aerosols.sel(lambda_scat=525)['scat_coeff']
ax = axs[1]
ds_sel.groupby(f"time.{freq}").mean().plot(
    label=ds_sel.attrs["description"],
    ax=axs[1]
)
ax.set_title('Total scattering coefficient at 525nm',loc='left')

# # Backscattering fraction
# ds_sel = ds_neph['BbsG0_S11']/ds_neph['BsG0_S11']
# ds_sel.groupby(f"time.{freq}").mean().plot(
#     label='scattering fraction',
#     ax=axs[1]
# )

# Backscatter fraction

# for ax in axs:
#     ax.legend()
plt.show()

### Compare with gas measurements

In [ ]:
## Read all GHG data
%autoreload 2
from input.read_wdc_data import AvailableData, create_data_reader

# File path
data_path = "../data/"

# if New data is added to ./data folder, adapt the dictionary in AvailableData
all_data = list(AvailableData)
print(all_data)

#####---------- TO ADAPT ---------------#####
selected_data = ['CO2', 'CO2_flask', 
                 'CO', 'CO_flask', 
                 'CH4', 'CH4_flask', 
                 'O3'
                 ] # define data to read in. If empty, all data is used 
## 
processing_kwargs = { 
    'FLASK_FLAG_CORR' : True # exclude flagged flask-data
}
#####-----------------------------------#####

datasets = [] # initialize list of all datasets 
# read in data
for sel in (selected_data if selected_data else all_data):
    #define where the data has to be read from
    data_reader =  create_data_reader(data_path=data_path,dataset=sel,**processing_kwargs) #creates an instance of the desired data_reader class
    print(f"Data from {data_reader.__class__.__name__} for {sel}:")

    # call the data-reading function on that instance: 
    data = data_reader.read_data() 
    # call the data-processing
    data = data_reader.process_data(data)

    # prepare merged dataset
    data = data.drop(columns='endtime') # problem when merging datasets (because of NaT?), so better remove endtime
    ds = data.to_xarray()
    ds = ds.assign_coords(dataset=sel)
    ds['species'] = data_reader.species
    ds['unit']  = np.unique(ds.unit.dropna(dim='time'))[0]
    datasets.append(ds)

# save all in one xarray dataset
ds_all = xr.concat(datasets,dim="dataset")


In [ ]:
## Figure for single species
tsel1 = "2020-01-01"
tsel2 = "2023-12-31"

species_ghg = "CO"
species_aer = 'black_carbon'

ghg_sel = ds_all.sel(time=slice(tsel1, tsel2), dataset=species_ghg, drop=True)
aer_sel = ds_aerosols.sel(lambda_abs=880)[species_aer]

fig, ax = plt.subplots(figsize=(6, 4))
alpha = 0.8
ms = 8

pl_ghg = ghg_sel["value"].plot(
    marker=".",
    ls="-",
    label=species_ghg,
    alpha=alpha,
    markeredgewidth=0,
    markersize=ms,
    ax=ax,
        color='C1'
)
plt.legend(loc='upper left')
ax2 = ax.twinx()
pl_aer = aer_sel.plot(
    marker=".",
    ls="-",
    label=species_aer,
    alpha=alpha,
    markeredgewidth=0,
    markersize=ms,
    ax=ax2,
    color='C0'
)
plt.legend()

mw_days = 5  # moving window days
mw = 3 * 8 * mw_days  #  moving windowplt.ylabel("CO2 (ppm)")
plt.xlabel("Time")
plt.legend()
plt.title("Mt. Kenya GAW station: \n COobservations and Black carbon")
form_xdate(ax, "Y", 1, "")
fig_format = "png"
plt.tight_layout()
# plt.savefig(fr'.\analyses\MKN_co2_gaw_cams_inv.{fig_format}',format=fig_format,dpi=400,transparent=False,facecolor='white')

# plt.savefig(f'.\\analyses\MKN_co2_gaw_cams_inv_mw{mw_days}d.{fig_format}',format=fig_format,dpi=400,transparent=False,facecolor='white')

### Scatter plot

In [ ]:
from sklearn.linear_model import LinearRegression

## Black carbon vs. CO for different months (using daily data)


x = ds_aerosols.sel(time=slice(tsel1,tsel2),lambda_abs=880)[species_aer].resample(time="D").mean()
y = ds_all.sel(time=slice(tsel1, tsel2), dataset=species_ghg, drop=True)['value'].resample(time="D").mean()

months = x.time.dt.month

# Define a colormap for months
colmap = tol_colors.tol_cmap("rainbow_discrete", 12)  # 12 colors for 12 months
colors = colmap(np.linspace(0, 1, 12))

fig, ax = plt.subplots(
    1, 1, figsize=(5,5), sharey=True, sharex=True, layout="constrained"
)

# all months scatter with linear fits
scatter = ax.scatter(x.values, y.values, c=months, cmap=colmap, alpha=1)
## add line fits:
# Iterate over unique months and fit a line for each month
for month, c in zip(range(1, 13), colors):
    # Filter data for the current month
    x_month = x.where(x.time.dt.month == month, drop=True)
    y_month = y.where(y.time.dt.month == month, drop=True)

    # remove nans
    y_month_non_nan = y_month.dropna(dim="time")
    x_month_non_nan = x_month.sel(time=y_month_non_nan.time.values, drop=True)

    x_month_non_nan = x_month_non_nan.dropna(dim="time")
    y_month_non_nan = y_month_non_nan.sel(time=x_month_non_nan.time.values, drop=True)

    if len(x_month) > 1:
        # Fit a linear regression model
        model = LinearRegression()
        model.fit(x_month_non_nan.values.reshape(-1, 1), y_month_non_nan.values)

        # Predictions
        x_pred = np.linspace(
            x.min(), x.max(), 2
        )  # create x-values for which to predict y-values
        y_pred = model.predict(x_pred.reshape(-1, 1))

        # Plot the line fit for the current month
        r2 = model.score(x_month_non_nan.values.reshape(-1, 1), y_month_non_nan.values)
        month_str = dt.datetime.strptime(str(month), "%m").strftime("%b")
        ax.plot(
            x_pred, y_pred, color=c, label=f"{month_str} ({r2:.2f})"
        )  # color=colmap(month / 13)
        # ax.plot(x_pred, y_pred, color=colmap(month / 13), label=f'Month {month}')
    else:
        print(f"no fit for month {month}")
    legend1 = ax.legend(
        loc="upper right", title="Months (R2)"
)
ax.add_artist(legend1)

ax.set_ylabel(f"CO")
ax.set_xlabel("Black Carbon")

# general figure properties
plt.setp(axs, box_aspect=1)  # get quadratic plots

plt.suptitle(
    "observed CO vs. black carbon \n at Mt. Kenya for different months"
)
#if save_fig:
    #plt.savefig(f"{dir_save}scatter_plots_fire_{obs_var}.pdf")
plt.show()